# 第 7 章 · 多 agent（agent-as-tool）

把**一个 agent 包装成工具**，给另一个 agent 调用。

和 Coordinator 的区别：那里是"父 agent 把对话转交给子 agent"，
这里是"父 agent 像调函数一样调子 agent，拿回一个返回值继续干自己的活"。

kernel 选 `Python (agentic-adk)`。

In [1]:
import sys
import uuid
from pathlib import Path

# 【适配】定位仓库根；后端（官方 / 中转站）由 .env 的 ADK_BACKEND 决定
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import adk_model, quiet_adk_sync_warnings, retry_call
quiet_adk_sync_warnings()

from google.adk.runners import InMemoryRunner
from google.genai import types


def run_once(agent, message):
    """跑一次，返回 (每个 agent 的最终答复, 结束时 session.state)。"""
    runner = InMemoryRunner(agent)
    sid = str(uuid.uuid4())
    runner.session_service.create_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    finals = []
    for ev in retry_call(lambda: list(runner.run(
            user_id="u", session_id=sid,
            new_message=types.Content(role="user", parts=[types.Part(text=message)]))), label="run "):
        if ev.is_final_response() and ev.content:
            t = getattr(ev.content, "text", None) or "".join(
                p.text for p in (ev.content.parts or []) if getattr(p, "text", None))
            finals.append((getattr(ev, "author", "?"), (t or "").strip()))
    sess = runner.session_service.get_session_sync(
        app_name=runner.app_name, user_id="u", session_id=sid)
    return finals, sess.state

from google.adk.agents import LlmAgent
from google.adk.tools import agent_tool


def generate_image(prompt: str) -> dict:
    """Generates an image based on a textual prompt.

    Args:
        prompt: A detailed description of the image to generate.

    Returns:
        A dictionary with the status and a short description of the image.
    """
    # 【适配】书里返回 mock 的 image bytes；这里返回文字描述，方便在 notebook 里看结果
    print(f"TOOL: Generating image for prompt: '{prompt}'")
    return {"status": "success", "description": f"已生成图片：{prompt}"}


image_generator_agent = LlmAgent(
    name="ImageGen",
    model=adk_model(),                      # 【适配】gemini-2.0-flash 已下线
    description="Generates an image based on a detailed text prompt.",
    instruction="你是图像生成专家。把用户的整个请求当作 prompt 传给 generate_image 工具，"
                "然后把工具返回的描述原样告诉我。",
    tools=[generate_image],
)

# ★ 本章新增 AgentTool：把一个 agent 变成另一个 agent 可调用的工具
# 【适配】书里还传了 description=...；ADK 2.13 不接受该参数，工具描述改由
#   被包装 agent 自己的 description 提供（见上面 ImageGen 的 description）
image_tool = agent_tool.AgentTool(agent=image_generator_agent)

artist_agent = LlmAgent(
    name="Artist",
    model=adk_model(),                      # 【适配】同上
    instruction="你是创意艺术家。先想一个有创意的画面描述，然后用 ImageGen 工具生成它。",
    tools=[image_tool],
)

finals, state = run_once(artist_agent, "画一只戴礼帽的猫")
for author, text in finals:
    print(f"\n--- [{author}] ---\n{text[:400]}")

TOOL: Generating image for prompt: '一只高贵的英国短毛猫，头戴一顶精致的黑色小礼帽，脖子上系着一个黑色的蝴蝶结领结。它端坐在一个温馨、光线柔和的书房里，背景是模糊的书架和陈旧的书籍。猫咪的毛发呈蓝灰色，质感蓬松，一双圆滚滚的金铜色眼睛平静而睿智地注视着镜头。整张照片色彩温暖，具有浅景深效果，背景有美丽的虚化，细节丰富，展现出一种复古绅士的优雅气质。'



--- [Artist] ---
为你画了一只优雅的“绅士猫”：

这是一只高贵的英国短毛猫，头戴一顶精致的黑色小礼帽，脖子上还系着一个黑色的蝴蝶结领结。它端坐在一个温馨、光线柔和的书房里，背景中隐约可见装满旧书的书架。猫咪拥有蓬松的蓝灰色毛发和一双圆滚滚的金铜色眼睛，眼神平静而睿智。整张照片色彩温暖，带有复古而优雅的质感。
